# 🛰️ Predictive Modeling for GNSS Satellite Clock & Ephemeris Errors
### A Step-by-Step Deep Learning Project for 7th Semester Students

---

## 🎯 What is this project about?

When your phone or a car's GPS tells you where you are, it is relying on signals from satellites orbiting Earth. These satellites carry **atomic clocks** and broadcast their precise position (called **ephemeris**) so your device can calculate distance using signal travel time.

**The problem:** Even atomic clocks drift slightly. Satellite orbits are also not perfectly predictable. These tiny errors — called **clock bias** and **ephemeris errors** — add up and degrade your GPS accuracy from centimeters to meters.

**Our goal:** Use Deep Learning (LSTM, Transformer) and classical ML (XGBoost, Kalman Filter) to **predict these errors before they happen**, so we can correct them in real time.

---

## 🌍 SDG Alignment
| SDG | How this project contributes |
|-----|-----------------------------|
| **SDG 9** — Industry, Innovation & Infrastructure | Better GNSS accuracy improves navigation, autonomous vehicles, 5G timing synchronization |
| **SDG 11** — Sustainable Cities & Communities | Precise positioning enables smart city sensors, traffic management, disaster response |
| **SDG 13** — Climate Action | Accurate GNSS supports atmospheric monitoring, sea-level rise tracking, glacier studies |
| **SDG 17** — Partnerships for the Goals | Built on open IGS global data — a true international public-good collaboration |

---

## 📚 What you will learn
1. How GNSS data is structured (SP3 files, RINEX format)
2. Time-series feature engineering (lag features, Fourier terms)
3. Building and training an LSTM from scratch in PyTorch
4. Building a Transformer encoder for time series
5. Using XGBoost for tabular time series
6. Implementing a Kalman Filter (classical approach)
7. Evaluating and comparing models (RMSE, MAE, R²)
8. Multi-step forecasting (predicting future epochs)

---

> 💡 **Note for students:** If the NASA CDDIS download fails (it requires registration), the notebook automatically switches to **synthetic data** that mimics real GNSS behavior perfectly. You can run the entire project without any external data!

---
## 📦 Section 1: Install & Import Libraries

> **What are we installing?**
> - `georinex` — parses RINEX navigation files (GNSS standard format)
> - `xgboost` — gradient boosted trees (fast, powerful for tabular data)
> - `torch` — PyTorch for building LSTM and Transformer models
> - `scikit-learn` — preprocessing (StandardScaler) and evaluation metrics
> - `scipy` — signal processing (Welch PSD for frequency analysis)
> - `matplotlib` / `seaborn` — plotting and visualization

In [ ]:
# Install all required packages quietly
# The -q flag suppresses verbose output so your screen doesn't get flooded
!pip install -q georinex xgboost scikit-learn pandas numpy matplotlib seaborn torch requests tqdm scipy

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# We organise imports into logical groups so the code is easy to read
# ─────────────────────────────────────────────────────────────────────────────

# Standard library
import os
import requests          # for downloading files from the internet
import gzip              # for decompressing .gz files
import shutil            # for file copy operations
import warnings
from datetime import datetime, timedelta

# Numerical & data
import numpy as np
import pandas as pd
from scipy import signal  # for Welch power spectral density

# Plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# PyTorch (Deep Learning)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Machine Learning (classical)
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL SETTINGS
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42                # random seed for reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

# Automatically use GPU if available in Colab (Runtime → Change runtime type → GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {device}')
print(f'   PyTorch version: {torch.__version__}')

# Plot style
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

---
## 📡 Section 2: Understanding GNSS Data

> ### What is an SP3 file?
> 
> The **SP3 (Standard Product 3)** format is the international standard for sharing precise satellite orbit and clock data. It is published by the **IGS (International GNSS Service)** — a global network of ~500 tracking stations that continuously monitor GPS, GLONASS, Galileo, and BeiDou satellites.
> 
> Each SP3 file contains, for every 15-minute epoch:
> ```
> * 2023  3  1  0  0  0.00000000         96         0.000000  0    0
> PG01  15347.741763  -3738.662921  21148.773946    123.456789   7  7  9  54
> PG02  20483.225489  13271.834021  14219.654321   -456.789012   7  7  9  54
> ```
> - `*` line = epoch timestamp
> - `P` line = Position (x, y, z in km) + Clock offset (microseconds)
> - `G01` = GPS satellite #1 (G=GPS, R=GLONASS, E=Galileo, C=BeiDou)
> 
> ### What is clock bias?
> 
> Think of it this way: GPS works like a **triangulation puzzle** — your receiver measures how long a signal took to arrive from 4+ satellites, multiplies by speed of light, and computes your position. Even a **1 nanosecond** clock error = **30 cm position error**! So predicting clock drift is critical.
> 
> ### What is ephemeris error?
> 
> Satellites broadcast where they *think* they are (broadcast ephemeris). IGS computes where they *actually* were (precise ephemeris). The difference = **ephemeris error**, measured in centimeters, in three axes:
> - **Radial** (towards/away from Earth center)
> - **Along-track** (in direction of motion)
> - **Cross-track** (perpendicular to orbit plane)

---
## 💾 Section 3: Download Real IGS SP3 Data

> **Where does the data come from?**
> 
> NASA's **CDDIS (Crustal Dynamics Data Information System)** archives all IGS products freely. We download **IGS Final SP3** files — these are the most accurate, published ~2 weeks after the observation date.
> 
> **GPS Week:** GPS time doesn't use calendar dates. It counts weeks since **January 6, 1980** (GPS epoch). For example, March 1, 2023 = GPS Week 2252, Day 3.
> 
> **What if download fails?** CDDIS requires registration for bulk downloads. If it fails, Section 4 auto-generates realistic synthetic data — the ML pipeline is identical either way.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Convert calendar date → GPS week + day-of-week
# GPS counts weeks from Jan 6, 1980 (the GPS epoch)
# This is needed to construct the correct SP3 filename
# ─────────────────────────────────────────────────────────────────────────────
def gps_week_day(dt):
    """
    Convert a Python datetime to GPS week number and day-of-week.
    
    Args:
        dt: datetime object
    Returns:
        (week, day) tuple — e.g. (2252, 3) for March 1, 2023
    """
    GPS_EPOCH = datetime(1980, 1, 6)   # GPS time started on this date
    delta = dt - GPS_EPOCH
    week = delta.days // 7             # integer division gives full weeks
    day  = delta.days % 7              # remainder gives day within week
    return week, day


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Download one SP3 file for a given date
# Files are stored compressed (.Z format) on NASA's server
# ─────────────────────────────────────────────────────────────────────────────
def download_sp3(date, save_dir='sp3_files'):
    """
    Download IGS final SP3 file for a given date from NASA CDDIS.
    Returns local file path if successful, None if failed.
    """
    os.makedirs(save_dir, exist_ok=True)
    week, day = gps_week_day(date)

    # SP3 filename convention: igs{week}{day}.sp3
    # e.g. igs225230.sp3 for GPS week 2252, day 3
    filename    = f'igs{week}{day}.sp3'
    gz_filename = filename + '.Z'          # compressed version
    local_path  = os.path.join(save_dir, filename)

    # Skip download if we already have the file cached
    if os.path.exists(local_path):
        print(f'  [cached] {filename}')
        return local_path

    # Construct download URL
    url = f'https://cddis.nasa.gov/archive/gnss/products/{week}/{gz_filename}'
    print(f'  Downloading {url}')

    try:
        # stream=True means we download in chunks (avoids memory issues for large files)
        r = requests.get(url, timeout=30, stream=True)
        if r.status_code == 200:
            gz_path = local_path + '.Z'
            with open(gz_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            # Decompress: .Z is Unix compress format, compatible with gzip module
            with gzip.open(gz_path, 'rb') as f_in:
                with open(local_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            os.remove(gz_path)   # clean up compressed file
            return local_path
        else:
            print(f'    HTTP {r.status_code} — registration may be required')
            return None
    except Exception as e:
        print(f'    Error: {e}')
        return None


# Download 7 days of SP3 data (1 week = 672 epochs at 15-min intervals)
print('Attempting to download 7 days of IGS SP3 data...')
start_date = datetime(2023, 3, 1)
dates      = [start_date + timedelta(days=i) for i in range(7)]
sp3_paths  = [download_sp3(d) for d in dates]
sp3_paths  = [p for p in sp3_paths if p is not None]  # filter failed downloads

print(f'\n✅ Successfully downloaded: {len(sp3_paths)} SP3 files')
if len(sp3_paths) == 0:
    print('⚠️  No files downloaded — switching to synthetic data in Section 4')

---
## 🔍 Section 4: Parse SP3 Files → Extract Usable Data

> **Why do we need to parse?**
> 
> SP3 is a plain text format. We need to read each line, identify the epoch markers (`*` lines) and position/clock records (`P` lines), and convert them into a pandas DataFrame that our ML models can work with.
> 
> After parsing, our DataFrame looks like:
> ```
>         epoch  prn      x_km       y_km       z_km   clock_us
> 0  2023-03-01  G01  15347.74  -3738.66  21148.77   0.000123
> 1  2023-03-01  G02  20483.22  13271.83  14219.65  -0.000456
> ```
> Clock is in **microseconds** — we'll convert to **nanoseconds** later (1 μs = 1000 ns).

In [ ]:
def parse_sp3(filepath):
    """
    Parse an IGS SP3 file into a structured DataFrame.
    
    SP3 line types we care about:
      '*' lines: epoch header (year, month, day, hour, minute, second)
      'P' lines: satellite position (x, y, z in km) + clock (microseconds)
    
    Clock values of 999999.999999 indicate missing/bad data → we filter these out.
    """
    records = []           # list to collect all valid records
    current_epoch = None   # track which epoch we're currently reading

    with open(filepath, 'r') as f:
        for line in f:
            # ── Epoch header line ──────────────────────────────────────────
            if line.startswith('*'):
                # Format: * YYYY MM DD HH MM SS.SSSSSSSSS
                parts = line.split()
                yr, mo, day  = int(parts[1]), int(parts[2]), int(parts[3])
                hr, mn       = int(parts[4]), int(parts[5])
                sec          = float(parts[6])
                current_epoch = datetime(yr, mo, day, hr, mn, int(sec))

            # ── Position + clock record ────────────────────────────────────
            elif line.startswith('P') and current_epoch is not None:
                parts = line.split()
                # PRN identifier, e.g. 'PG01' → 'G01'
                prn = parts[0][1:]
                try:
                    x   = float(parts[1])   # X coordinate (km)
                    y   = float(parts[2])   # Y coordinate (km)
                    z   = float(parts[3])   # Z coordinate (km)
                    clk = float(parts[4])   # clock offset (microseconds)

                    # Filter: 999999 is the SP3 sentinel for bad/missing data
                    if abs(clk) < 999999:
                        records.append({
                            'epoch': current_epoch,
                            'prn': prn,
                            'x_km': x, 'y_km': y, 'z_km': z,
                            'clock_us': clk
                        })
                except (ValueError, IndexError):
                    pass   # skip malformed lines silently

    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# Parse all downloaded SP3 files and concatenate into one big DataFrame
# ─────────────────────────────────────────────────────────────────────────────
dfs = []
for path in sp3_paths:
    try:
        df = parse_sp3(path)
        dfs.append(df)
        print(f'  Parsed {os.path.basename(path)}: {len(df):,} records')
    except Exception as e:
        print(f'  Failed {path}: {e}')

if dfs:
    # Combine all days into one DataFrame
    data = pd.concat(dfs, ignore_index=True)
    data.sort_values(['prn', 'epoch'], inplace=True)
    data.reset_index(drop=True, inplace=True)

    # Convert clock from microseconds → nanoseconds (1 μs = 1000 ns)
    # We use nanoseconds because 1 ns = 30 cm position error → more interpretable unit
    data['clock_ns'] = data['clock_us'] * 1000.0

    # Placeholders for ephemeris error (computed from broadcast vs precise)
    data['ephem_radial_cm'] = np.nan
    data['ephem_along_cm']  = np.nan
    data['ephem_cross_cm']  = np.nan
    data['kp_index']        = np.nan

    print(f'\n✅ Total records : {len(data):,}')
    print(f'   Satellites    : {data["prn"].nunique()} unique PRNs')
    print(f'   Time range    : {data["epoch"].min()} → {data["epoch"].max()}')
    data.head()

---
## 🧪 Section 5: Synthetic Data Generator (Auto-Fallback)

> ### Why synthetic data?
> 
> Real GNSS clock errors follow well-understood physics:
> - A **linear drift** term (clocks speed up or slow down over time)
> - A **sinusoidal component** matching the orbital period (~12 hours for GPS)
> - **Random walk noise** (tiny random perturbations accumulating)
> - Occasional jumps from satellite maneuvers
> 
> Our synthetic model captures all these components:
> ```
> clock(t) = bias₀ + drift×t + A×sin(2π×t/T) + B×cos(4π×t/T) + noise
> ```
> 
> This is called a **clock model** and is literally what GNSS engineers use in production systems!
> 
> **Kp index:** A measure of geomagnetic activity (0–9 scale). High Kp = solar storm = worse GPS errors. We include it as a feature because space weather genuinely affects satellite clocks.

In [ ]:
def generate_synthetic_gnss(n_epochs=2016, n_sats=8, dt_minutes=15):
    """
    Generate realistic synthetic GNSS clock bias and ephemeris error time series.
    
    Clock model (nanoseconds):
        clock(t) = bias₀ + drift×t + A₁×sin(2πt/T) + A₂×cos(4πt/T) + noise
    
    Ephemeris model (centimeters, 3-axis):
        radial/along/cross each follow their own sinusoidal + noise model
    
    Args:
        n_epochs   : number of 15-minute samples (2016 = exactly 3 weeks)
        n_sats     : number of simulated satellites
        dt_minutes : sampling interval
    """
    epochs = [datetime(2023, 3, 1) + timedelta(minutes=i * dt_minutes)
              for i in range(n_epochs)]
    sats   = [f'G{i:02d}' for i in range(1, n_sats + 1)]
    records = []

    for prn in sats:
        # ── Per-satellite physical parameters (randomised for realism) ──────
        bias0         = np.random.uniform(-80, 80)      # initial clock offset (ns)
        drift         = np.random.uniform(-2e-3, 2e-3)  # drift rate (ns/epoch)
        amp1          = np.random.uniform(2, 10)        # primary sinusoid amplitude (ns)
        amp2          = np.random.uniform(0.5, 3)       # secondary sinusoid amplitude (ns)
        period        = np.random.uniform(250, 580)     # ~orbital period in epochs
        noise_std     = np.random.uniform(0.2, 0.8)    # Gaussian noise std (ns)
        ephem_scale   = np.random.uniform(0.5, 3.0)    # ephemeris error magnitude (cm)

        for i, ep in enumerate(epochs):
            t = float(i)   # time index

            # ── Clock bias model ─────────────────────────────────────────────
            clock_ns = (
                bias0                                         # constant offset
                + drift * t                                   # linear drift
                + amp1 * np.sin(2 * np.pi * t / period)      # fundamental orbital period
                + amp2 * np.cos(4 * np.pi * t / period)      # second harmonic
                + np.random.normal(0, noise_std)             # white noise
            )

            # ── Ephemeris errors (3 axes, centimeters) ────────────────────
            # Each axis has a different phase offset to simulate independent errors
            radial = ephem_scale * (
                0.6 * np.sin(2 * np.pi * t / period + 0.3)
                + np.random.normal(0, 0.12)
            )
            along = ephem_scale * (
                1.0 * np.cos(2 * np.pi * t / period + 1.1)
                + np.random.normal(0, 0.18)
            )
            cross = ephem_scale * (
                0.4 * np.sin(2 * np.pi * t / period + 2.3)
                + np.random.normal(0, 0.08)
            )

            # ── Space weather: Kp index (0–9 scale) ──────────────────────
            # Kp is typically low (2-3) with occasional storms (Kp > 5)
            # We model it as a slowly varying sinusoid + noise, clipped to [0,9]
            kp = np.clip(
                3.0 + 2.0 * np.sin(2 * np.pi * t / 1440) + np.random.normal(0, 0.6),
                0, 9
            )

            records.append({
                'epoch': ep, 'prn': prn,
                'clock_ns': clock_ns,
                'ephem_radial_cm': radial,
                'ephem_along_cm': along,
                'ephem_cross_cm': cross,
                'kp_index': kp
            })

    df = pd.DataFrame(records)
    df.sort_values(['prn', 'epoch'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Use synthetic data if real data could not be downloaded
# ─────────────────────────────────────────────────────────────────────────────
if not dfs:
    print('⚠️  No real SP3 data — generating synthetic GNSS data...')
    data = generate_synthetic_gnss(n_epochs=2016, n_sats=8)
    print(f'✅ Synthetic data ready:')
    print(f'   Satellites   : {data["prn"].nunique()}')
    print(f'   Total records: {len(data):,}')
    print(f'   Time span    : {data["epoch"].min().date()} → {data["epoch"].max().date()}')
    print(f'   Clock range  : {data["clock_ns"].min():.2f} to {data["clock_ns"].max():.2f} ns')

print('\nFirst 5 rows:')
data.head()

---
## 📊 Section 6: Exploratory Data Analysis (EDA)

> ### Why EDA before modeling?
> 
> Before throwing data into a neural network, you **always** want to understand:
> 1. **Distribution** — is the data normally distributed? Skewed? Bimodal?
> 2. **Trends** — is there a linear drift? Seasonal patterns?
> 3. **Frequency content** — what periodicities exist? (tells us what Fourier features to add)
> 4. **Stationarity** — does the mean/variance change over time? (important for RNNs)
> 5. **Correlation** — how much does the current value depend on past values? (tells us lag length)
> 
> ### What to look for in the plots:
> - **Time series plot:** You should see the 12-hour orbital sinusoid on top of slow drift
> - **Histogram:** Should be roughly bell-shaped (Gaussian), possibly slightly skewed
> - **PSD:** Should show a clear peak around the orbital frequency (~2 cycles/day)
> - **Autocorrelation:** Should decay slowly — tells us the signal is strongly autocorrelated (good for LSTM!)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Pick one satellite for detailed per-satellite analysis
# We'll use G01 (GPS satellite #1) as our primary example
# ─────────────────────────────────────────────────────────────────────────────
TARGET_PRN = data['prn'].unique()[0]   # e.g. 'G01'
sat_df = data[data['prn'] == TARGET_PRN].copy().reset_index(drop=True)
print(f'Analyzing satellite: {TARGET_PRN} | {len(sat_df)} epochs')

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Plot 1: Clock bias time series ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])  # wide, top-left
ax1.plot(sat_df['epoch'], sat_df['clock_ns'], lw=0.8, color='steelblue')
ax1.set_title(f'Satellite {TARGET_PRN} — clock bias over time', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Clock bias (ns)')
ax1.tick_params(axis='x', rotation=20)

# ── Plot 2: Clock bias histogram ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])   # top-right
ax2.hist(sat_df['clock_ns'], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
ax2.set_title('Distribution of clock bias', fontsize=12)
ax2.set_xlabel('Clock bias (ns)')
ax2.set_ylabel('Count')

# ── Plot 3: Ephemeris radial error ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])  # wide, bottom-left
if sat_df['ephem_radial_cm'].notna().any():
    ax3.plot(sat_df['epoch'], sat_df['ephem_radial_cm'], lw=0.8, color='coral')
    ax3.set_title(f'Satellite {TARGET_PRN} — radial ephemeris error', fontsize=12)
    ax3.set_ylabel('Radial error (cm)')
    ax3.tick_params(axis='x', rotation=20)
else:
    ax3.text(0.5, 0.5,
             'With real SP3 data:\nSubtract broadcast ephemeris\nfrom precise SP3 positions',
             ha='center', va='center', transform=ax3.transAxes, fontsize=11,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax3.set_title('Ephemeris radial error (requires broadcast nav file)', fontsize=12)

# ── Plot 4: Power Spectral Density ──────────────────────────────────────────
# PSD shows us which frequencies dominate the signal
# Peak at ~1/576 (cycles/sample) = 12-hour orbital period at 15-min sampling
ax4 = fig.add_subplot(gs[1, 2])   # bottom-right
series = sat_df['clock_ns'].fillna(method='ffill').values
nperseg = min(512, len(series) // 4)
f_psd, psd = signal.welch(series, nperseg=nperseg)
ax4.semilogy(f_psd, psd, color='purple', lw=0.9)
ax4.set_title('Power spectral density', fontsize=12)
ax4.set_xlabel('Frequency (cycles/sample)')
ax4.set_ylabel('PSD (log scale)')

fig.suptitle(f'EDA — Satellite {TARGET_PRN} Clock & Orbit Analysis', fontsize=14, fontweight='bold')
plt.savefig('eda_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved as eda_plot.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Multi-satellite comparison: see how different sats behave
# This motivates WHY we need per-satellite models (each clock behaves differently)
# ─────────────────────────────────────────────────────────────────────────────
all_prns = data['prn'].unique()[:6]   # plot up to 6 satellites
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=False)
axes = axes.flatten()

for i, prn in enumerate(all_prns):
    s = data[data['prn'] == prn]
    axes[i].plot(s['epoch'], s['clock_ns'], lw=0.7)
    axes[i].set_title(f'Satellite {prn}', fontsize=11)
    axes[i].set_ylabel('Clock bias (ns)')
    axes[i].tick_params(axis='x', rotation=20, labelsize=8)

fig.suptitle('Clock bias for 6 satellites — each has its own drift signature',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Observation: Each satellite clock has a unique bias, drift rate and sinusoidal pattern.')
print('This is why per-satellite modeling (or PRN embedding) is important!')

---
## 🔧 Section 7: Feature Engineering — Turning Raw Data into ML Inputs

> ### This is the most important section for ML beginners!
> 
> Raw time series data is just a sequence of numbers. ML models need **features** — inputs that capture the patterns we want to learn. Here's what we create:
> 
> ---
> 
> **1. Lag Features (`lag_1`, `lag_2`, ..., `lag_12`)**
> 
> The most recent past values of clock bias. If `clock(t)` = current value, then `lag_1` = `clock(t-1)` (15 minutes ago), `lag_2` = `clock(t-2)` (30 minutes ago), etc.
> 
> *Why?* GNSS clocks have very high autocorrelation — yesterday's value is the best predictor of today's value.
> 
> **2. Difference Features (`diff1`, `diff2`)**
> 
> First difference = `clock(t) - clock(t-1)` (rate of change = "velocity")
> Second difference = velocity change = "acceleration" of clock drift
> 
> *Why?* These capture the drift rate independently of absolute clock value.
> 
> **3. Fourier Features (`sin_1`, `cos_1`, `sin_2`, `cos_2`, ...)**
> 
> We encode the orbital period mathematically: `sin(2π×t/T)`, `cos(2π×t/T)`
> 
> GPS orbital period ≈ 12 hours = 48 epochs (at 15-min intervals)
> 
> *Why?* Neural networks struggle to learn periodic patterns from raw time indices. Fourier features give the model a "head start" by explicitly encoding the periodicity.
> 
> **4. Cyclical Time Encoding**
> 
> Hour-of-day and day-of-week encoded as `sin/cos` pairs.
> 
> *Why?* Time of day matters because solar radiation pressure on satellites changes with Sun angle. Using `sin(hour)` instead of raw hour ensures hour 23 and hour 0 are close to each other.
> 
> **5. Rolling Statistics**
> 
> Rolling mean and std over short windows (1h, 2h, 6h)
> 
> *Why?* Captures the recent average level and volatility of the clock.
> 
> **6. Kp Index**
> 
> Geomagnetic activity index — affects satellite clocks during solar storms.
> 
> **Target variable:** `clock_ns` at time `t+1` (one step ahead prediction)

In [ ]:
def build_features(df, target_col='clock_ns', n_lags=16, fourier_k=4, dt_minutes=15):
    """
    Build a rich ML feature matrix for a single satellite time series.

    Args:
        df          : single-satellite DataFrame (sorted by epoch)
        target_col  : name of the column we want to predict
        n_lags      : how many past timesteps to include as features (lookback)
        fourier_k   : number of Fourier harmonics to include
        dt_minutes  : sampling interval in minutes

    Returns:
        DataFrame with all features + 'target' column
    """
    df = df.copy().reset_index(drop=True)
    t  = np.arange(len(df), dtype=float)   # integer time index

    # ── 1. Fourier features (orbital period encoding) ────────────────────────
    # GPS orbital period ≈ 12 hours = 720 minutes / 15 min per epoch = 48 epochs
    # We use a few harmonics to capture non-sinusoidal patterns
    orbital_period_epochs = (12 * 60) / dt_minutes   # = 48 for 15-min data
    for k in range(1, fourier_k + 1):
        df[f'sin_{k}'] = np.sin(2 * np.pi * k * t / orbital_period_epochs)
        df[f'cos_{k}'] = np.cos(2 * np.pi * k * t / orbital_period_epochs)

    # ── 2. Time-of-day features (cyclical encoding) ─────────────────────────
    # sin/cos encoding maps circular time onto a circle in 2D space
    # so the model knows that hour 23 and hour 0 are adjacent
    hour_frac = df['epoch'].dt.hour + df['epoch'].dt.minute / 60.0
    df['hour_sin'] = np.sin(2 * np.pi * hour_frac / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * hour_frac / 24.0)

    # Day-of-week (Mon=0, Sun=6)
    df['dow_sin'] = np.sin(2 * np.pi * df['epoch'].dt.dayofweek / 7.0)
    df['dow_cos'] = np.cos(2 * np.pi * df['epoch'].dt.dayofweek / 7.0)

    # Day of year (captures annual solar/atmospheric effects)
    df['doy_sin'] = np.sin(2 * np.pi * df['epoch'].dt.dayofyear / 365.0)
    df['doy_cos'] = np.cos(2 * np.pi * df['epoch'].dt.dayofyear / 365.0)

    # ── 3. Lag features ──────────────────────────────────────────────────────
    # shift(1) gives previous timestep; shift(2) gives two timesteps back, etc.
    for lag in range(1, n_lags + 1):
        df[f'lag_{lag}'] = df[target_col].shift(lag)

    # ── 4. Difference features (rate of change) ──────────────────────────────
    df['diff1'] = df[target_col].diff(1)   # first difference (velocity)
    df['diff2'] = df[target_col].diff(2)   # second difference (acceleration)
    df['diff4'] = df[target_col].diff(4)   # 1-hour difference

    # ── 5. Rolling statistics ────────────────────────────────────────────────
    # min_periods=1 avoids NaN at the start of the series
    for w in [4, 8, 16, 48]:              # 1h, 2h, 4h, 12h windows
        df[f'roll_mean_{w}'] = df[target_col].rolling(w, min_periods=1).mean()
        df[f'roll_std_{w}']  = df[target_col].rolling(w, min_periods=1).std()

    # ── 6. Exponential moving average (EMA) ──────────────────────────────────
    # EMA gives more weight to recent values — useful for tracking drift
    df['ema_8']  = df[target_col].ewm(span=8,  adjust=False).mean()
    df['ema_48'] = df[target_col].ewm(span=48, adjust=False).mean()

    # ── 7. Space weather ─────────────────────────────────────────────────────
    if 'kp_index' in df.columns and df['kp_index'].notna().any():
        df['kp_index'] = df['kp_index'].fillna(df['kp_index'].median())
        # Lag Kp too — solar storms affect clocks with a delay
        df['kp_lag1'] = df['kp_index'].shift(1)
        df['kp_lag4'] = df['kp_index'].shift(4)

    # ── 8. Target: predict ONE step ahead ────────────────────────────────────
    # shift(-1) moves values back, so row[t] gets the value from t+1
    df['target'] = df[target_col].shift(-1)

    # Drop rows with any NaN (created by shift/rolling at edges)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# Apply feature engineering to our target satellite
feat_df = build_features(sat_df, n_lags=16, fourier_k=4)

# Identify all feature columns (exclude metadata and target)
EXCLUDE = {'epoch', 'prn', 'target', 'clock_ns', 'clock_us',
           'x_km', 'y_km', 'z_km', 'ephem_radial_cm',
           'ephem_along_cm', 'ephem_cross_cm'}
feature_cols = [c for c in feat_df.columns if c not in EXCLUDE]

print(f'✅ Feature matrix shape : {feat_df[feature_cols].shape}')
print(f'   Number of features   : {len(feature_cols)}')
print(f'   Feature names        : {feature_cols}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise feature correlations with the target
# This shows which features are most predictive of the next clock value
# ─────────────────────────────────────────────────────────────────────────────
corr_with_target = feat_df[feature_cols + ['target']].corr()['target'].drop('target')
corr_sorted = corr_with_target.abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 5))
colors = ['#2196F3' if corr_with_target[c] > 0 else '#F44336' for c in corr_sorted.index]
plt.barh(corr_sorted.index[::-1], corr_sorted.values[::-1], color=colors[::-1])
plt.xlabel('|Correlation with target|')
plt.title('Top 20 features by correlation with next-step clock bias', fontsize=12)
plt.tight_layout()
plt.show()
print('Expected: lag_1, lag_2 should dominate — the signal is highly autocorrelated')

---
## ✂️ Section 8: Train / Validation / Test Split

> ### Why chronological splitting? (NO random shuffling!)
> 
> For time series, **you must never randomly shuffle** before splitting. Why? Because:
> - Random split = training on future data, testing on past → **data leakage**
> - This gives artificially high accuracy that won't generalise to real deployment
> 
> We use a strict chronological 70/15/15 split:
> ```
> |────── Train (70%) ──────|── Val (15%) ──|── Test (15%) ──|
> t=0                   t=0.7            t=0.85           t=1.0
> ```
> 
> **StandardScaler:** Subtracts mean and divides by std from the *training set only*. We then apply the same transformation to val and test — this prevents the model from "peeking" at val/test statistics.
> 
> **Sequence creation (for LSTM/Transformer):** We reshape the flat feature matrix into overlapping windows of length `SEQ_LEN`:
> ```
> Input shape: (N - SEQ_LEN, SEQ_LEN, n_features)
> Output shape: (N - SEQ_LEN,)
> ```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Extract feature matrix X and target vector y
# ─────────────────────────────────────────────────────────────────────────────
X = feat_df[feature_cols].values.astype(np.float32)
y = feat_df['target'].values.astype(np.float32)

N = len(X)
train_end = int(N * 0.70)   # 70% for training
val_end   = int(N * 0.85)   # 15% for validation, 15% for test

# Chronological split — NO shuffle!
X_train, y_train = X[:train_end],        y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],          y[val_end:]

print(f'Split sizes:')
print(f'  Train : {X_train.shape}  ({train_end} samples = {train_end*15/60:.0f} hours)')
print(f'  Val   : {X_val.shape}')
print(f'  Test  : {X_test.shape}')

# ─────────────────────────────────────────────────────────────────────────────
# Scale features: fit ONLY on training data, then transform all sets
# This is critical — using val/test stats would be data leakage
# ─────────────────────────────────────────────────────────────────────────────
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on train
X_val_s   = scaler.transform(X_val)         # transform only (same parameters)
X_test_s  = scaler.transform(X_test)        # transform only (same parameters)

print(f'\nScaling applied: mean={scaler.mean_[:3].round(4)} ... (first 3 features)')
print(f'                  std ={scaler.scale_[:3].round(4)} ...')

# ─────────────────────────────────────────────────────────────────────────────
# Create sequences for LSTM / Transformer
# We use a lookback window of SEQ_LEN timesteps
# ─────────────────────────────────────────────────────────────────────────────
SEQ_LEN = 32   # 32 × 15 min = 8 hours of history fed to each prediction

def make_sequences(X_flat, y_flat, seq_len):
    """
    Convert a flat (N, F) feature matrix to overlapping 3D windows (N-L, L, F)
    where L = seq_len.
    
    Example: if N=100, F=40, seq_len=10
      Output X shape: (90, 10, 40)
      Output y shape: (90,)
      X[i] contains the feature rows from i to i+seq_len
      y[i] is the target at position i+seq_len
    """
    Xs, ys = [], []
    for i in range(len(X_flat) - seq_len):
        Xs.append(X_flat[i : i + seq_len])   # window of seq_len rows
        ys.append(y_flat[i + seq_len])        # target is the NEXT value after window
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_tr_seq, y_tr_seq = make_sequences(X_train_s, y_train, SEQ_LEN)
X_vl_seq, y_vl_seq = make_sequences(X_val_s,   y_val,   SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(X_test_s,  y_test,  SEQ_LEN)

print(f'\nSequence shapes (for LSTM/Transformer):')
print(f'  Train : {X_tr_seq.shape}  → (samples, seq_len, features)')
print(f'  Val   : {X_vl_seq.shape}')
print(f'  Test  : {X_te_seq.shape}')

# ─────────────────────────────────────────────────────────────────────────────
# Create PyTorch DataLoaders (handles batching and shuffling for training)
# We shuffle training sequences (individual windows are independent)
# but NOT val/test (keep chronological order for evaluation)
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE = 64

def make_loader(X_seq, y_seq, shuffle=False, batch_size=BATCH_SIZE):
    ds = TensorDataset(
        torch.from_numpy(X_seq),
        torch.from_numpy(y_seq)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(device=='cuda'))

train_loader = make_loader(X_tr_seq, y_tr_seq, shuffle=True)
val_loader   = make_loader(X_vl_seq, y_vl_seq, shuffle=False)
test_loader  = make_loader(X_te_seq, y_te_seq, shuffle=False)

print(f'\nDataLoaders ready — batch size: {BATCH_SIZE}')
print(f'  Training batches: {len(train_loader)}')

---
## 🌿 Section 9: Model 1 — XGBoost (Gradient Boosted Trees)

> ### What is XGBoost?
> 
> XGBoost builds an **ensemble of decision trees** sequentially, where each new tree corrects the errors of all previous trees. This is called **gradient boosting** — we fit each tree to the residual (error) of the current ensemble.
> 
> **Why use it for time series?**
> - Extremely fast to train (minutes vs hours for DL)
> - Handles tabular features natively (doesn't need sequences)
> - Great baseline — if DL doesn't beat XGBoost, your architecture may be wrong
> - Built-in early stopping prevents overfitting
> 
> **Key hyperparameters:**
> - `n_estimators` — max number of trees (early stopping controls actual number)
> - `max_depth` — depth of each tree (6 = balanced complexity)
> - `learning_rate` — how much each tree contributes (smaller = better, but needs more trees)
> - `subsample` — fraction of training data used per tree (prevents overfitting)
> - `colsample_bytree` — fraction of features used per tree (like dropout for trees)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# XGBoost uses the FLAT (non-sequence) features — no need for 3D input
# early_stopping_rounds: stop if val RMSE doesn't improve for 30 rounds
# ─────────────────────────────────────────────────────────────────────────────
print('Training XGBoost...')

xgb_model = xgb.XGBRegressor(
    n_estimators       = 600,      # max number of trees
    max_depth          = 6,        # tree depth (6 is a good default)
    learning_rate      = 0.04,     # shrinkage — smaller = more robust
    subsample          = 0.8,      # 80% of data per tree (bagging)
    colsample_bytree   = 0.8,      # 80% of features per tree
    min_child_weight   = 3,        # min samples in leaf (regularisation)
    reg_lambda         = 1.0,      # L2 regularisation
    early_stopping_rounds = 30,    # stop if val doesn't improve
    eval_metric        = 'rmse',   # metric for early stopping
    random_state       = SEED,
    n_jobs             = -1,       # use all CPU cores
    verbosity          = 0
)

# Train with early stopping on validation set
xgb_model.fit(
    X_train_s, y_train,
    eval_set       = [(X_val_s, y_val)],
    verbose        = 50   # print every 50 rounds
)

# Predict on test set
xgb_pred_test = xgb_model.predict(X_test_s)

# ── Evaluate ──────────────────────────────────────────────────────────────
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred_test))
xgb_mae  = mean_absolute_error(y_test, xgb_pred_test)
xgb_r2   = r2_score(y_test, xgb_pred_test)

print(f'\n── XGBoost Test Results ──────────────────')
print(f'  RMSE : {xgb_rmse:.4f} ns  ← target: as low as possible')
print(f'  MAE  : {xgb_mae:.4f} ns')
print(f'  R²   : {xgb_r2:.4f}     ← 1.0 = perfect, 0.0 = baseline mean')
print(f'  Best round: {xgb_model.best_iteration}')

---
## 🧠 Section 10: Model 2 — Bidirectional LSTM (PyTorch)

> ### What is an LSTM and why is it good for GNSS?
> 
> **LSTM (Long Short-Term Memory)** is a type of Recurrent Neural Network (RNN) designed to capture long-range dependencies in sequences. It solves the **vanishing gradient problem** that made vanilla RNNs useless for long sequences.
> 
> **Key components:**
> - **Cell state `c_t`** — the "memory tape" that passes information across time steps
> - **Forget gate** — decides what old information to erase from memory
> - **Input gate** — decides what new information to write to memory
> - **Output gate** — decides what to read from memory as the hidden state `h_t`
> 
> **Why bidirectional?** A standard LSTM only sees past timesteps. A `BiLSTM` runs two LSTMs: one forward (past → future) and one backward (future → past), then concatenates their hidden states. For GNSS, where the signal is periodic, the backward pass helps recognize the phase of the current orbital cycle.
> 
> **Architecture:**
> ```
> Input: (batch, SEQ_LEN=32, n_features=~50)
>   ↓
> BiLSTM (128 hidden × 2 directions = 256) × 2 layers
>   ↓
> Take last timestep's output: (batch, 256)
>   ↓
> Linear(256 → 64) → ReLU → Dropout(0.1)
>   ↓
> Linear(64 → 1) → scalar prediction
> ```

In [ ]:
class LSTMPredictor(nn.Module):
    """
    Stacked Bidirectional LSTM for GNSS clock error prediction.

    Architecture:
        Input → BiLSTM (2 layers) → FC head → scalar output

    Args:
        input_size  : number of input features (= len(feature_cols))
        hidden_size : LSTM hidden units per direction (default 128)
        num_layers  : number of stacked LSTM layers (default 2)
        dropout     : dropout between LSTM layers (default 0.2)
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMPredictor, self).__init__()

        # Bidirectional LSTM — hidden_size is PER direction
        # So the output dim = hidden_size × 2 (forward + backward)
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,       # input shape: (batch, seq, features)
            dropout     = dropout if num_layers > 1 else 0.0,  # between layers
            bidirectional = True
        )

        # Fully connected prediction head
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),  # × 2 because bidirectional
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)                 # final scalar output
        )

    def forward(self, x):
        """
        Forward pass.
        x: (batch_size, seq_len, input_size)
        returns: (batch_size,) — one prediction per sample
        """
        # lstm_out: (batch, seq_len, hidden_size * 2)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # We only need the output at the LAST timestep
        # lstm_out[:, -1, :] → (batch, hidden_size * 2)
        last_out = lstm_out[:, -1, :]

        # Pass through FC head → scalar
        return self.fc(last_out).squeeze(-1)  # (batch,)


# ─────────────────────────────────────────────────────────────────────────────
# Initialise model, optimiser, scheduler, loss function
# ─────────────────────────────────────────────────────────────────────────────
n_features = X_tr_seq.shape[2]   # number of input features

lstm_model = LSTMPredictor(
    input_size  = n_features,
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.2
).to(device)

# Adam: adaptive learning rate optimiser — works well out of the box for RNNs
lstm_opt = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

# ReduceLROnPlateau: halve LR when val loss stops improving for 5 epochs
lstm_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    lstm_opt, patience=5, factor=0.5, verbose=True
)

# Mean Squared Error loss — standard for regression
criterion = nn.MSELoss()

total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f'LSTM model architecture:')
print(lstm_model)
print(f'\nTotal trainable parameters: {total_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LSTM Training Loop
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS = 60
lstm_train_losses = []
lstm_val_losses   = []
best_val_loss     = float('inf')
best_lstm_path    = 'best_lstm.pt'

print(f'Training LSTM for up to {EPOCHS} epochs (early stopping via scheduler)...\n')

for epoch in range(1, EPOCHS + 1):
    # ── Training phase ───────────────────────────────────────────────────────
    lstm_model.train()   # enable dropout, batch norm
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        lstm_opt.zero_grad()            # clear gradients from previous step
        preds = lstm_model(X_batch)     # forward pass
        loss  = criterion(preds, y_batch)  # compute loss
        loss.backward()                 # backpropagation

        # Gradient clipping: cap gradient norm at 1.0
        # This prevents exploding gradients — common in RNNs with long sequences
        nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)

        lstm_opt.step()                 # update weights
        train_loss += loss.item()

    train_loss /= len(train_loader)    # average over batches

    # ── Validation phase ─────────────────────────────────────────────────────
    lstm_model.eval()    # disable dropout
    val_loss = 0.0

    with torch.no_grad():   # no gradient computation needed for evaluation
        for X_batch, y_batch in val_loader:
            preds    = lstm_model(X_batch.to(device))
            val_loss += criterion(preds, y_batch.to(device)).item()

    val_loss /= len(val_loader)

    lstm_train_losses.append(train_loss)
    lstm_val_losses.append(val_loss)

    # Step the scheduler — may reduce LR
    lstm_sched.step(val_loss)

    # Save best model checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(lstm_model.state_dict(), best_lstm_path)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train MSE: {train_loss:.5f} | '
              f'Val MSE: {val_loss:.5f} | '
              f'LR: {lstm_opt.param_groups[0]["lr"]:.2e}')

# ── Load best checkpoint and evaluate on test set ────────────────────────────
lstm_model.load_state_dict(torch.load(best_lstm_path))
lstm_model.eval()

lstm_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        out = lstm_model(X_batch.to(device)).cpu().numpy()
        lstm_preds.extend(out)

lstm_preds = np.array(lstm_preds, dtype=np.float32)

lstm_rmse = np.sqrt(mean_squared_error(y_te_seq, lstm_preds))
lstm_mae  = mean_absolute_error(y_te_seq, lstm_preds)
lstm_r2   = r2_score(y_te_seq, lstm_preds)

print(f'\n── LSTM Test Results ─────────────────────')
print(f'  RMSE : {lstm_rmse:.4f} ns')
print(f'  MAE  : {lstm_mae:.4f} ns')
print(f'  R²   : {lstm_r2:.4f}')

---
## ⚡ Section 11: Model 3 — Temporal Transformer

> ### Why Transformers for time series?
> 
> Transformers, originally designed for NLP ("Attention is All You Need", 2017), have proven excellent for time series because:
> - **Self-attention** computes relationships between ALL pairs of timesteps simultaneously
> - LSTMs process sequentially (slow); Transformers process in parallel (fast on GPU)
> - Self-attention can directly model the 12-hour orbital periodicity by attending to timesteps 48 steps apart
> 
> **Architecture for our task:**
> ```
> Input: (batch, SEQ_LEN, n_features)
>   ↓
> Linear projection → d_model=64 embedding
>   ↓
> + Positional encoding (sine/cosine, gives order information)
>   ↓
> TransformerEncoder × 3 layers
>   Each layer: MultiHeadAttention(heads=4) + FeedForward(d=256) + LayerNorm
>   ↓
> Take [CLS] token or mean-pool: (batch, d_model)
>   ↓
> GELU → Linear → scalar prediction
> ```
> 
> **Positional Encoding** — Transformers have no built-in notion of order (unlike LSTMs). We add sinusoidal positional encodings to inject temporal position information:
> ```
> PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
> PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
> ```

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding from 'Attention is All You Need' (Vaswani et al. 2017).
    Adds position information to the token embeddings so the model knows
    which timestep each feature vector belongs to.
    """
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Precompute the positional encoding matrix
        pe  = torch.zeros(max_len, d_model)   # shape: (max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)

        # Compute the frequency denominators: 10000^(2i/d_model)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)   # even dimensions: sine
        pe[:, 1::2] = torch.cos(pos * div)   # odd dimensions: cosine

        # Register as buffer (not a parameter — not updated by optimizer)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # Add positional encoding (broadcast across batch)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class GNSSTransformer(nn.Module):
    """
    Transformer encoder for GNSS clock error time-series prediction.

    Architecture:
        Input projection → Positional encoding → TransformerEncoder → FC head

    Args:
        input_size : number of input features
        d_model    : embedding dimension inside the Transformer (default 64)
        nhead      : number of attention heads (must divide d_model evenly)
        num_layers : number of Transformer encoder layers
        dim_ff     : feedforward network hidden size (d_model × 4 is standard)
        dropout    : dropout probability
    """
    def __init__(self, input_size, d_model=64, nhead=4,
                 num_layers=3, dim_ff=256, dropout=0.1):
        super().__init__()

        # Project raw features to d_model dimensional embedding space
        self.input_proj = nn.Linear(input_size, d_model)

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # Transformer encoder: stack of self-attention + feedforward layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_ff,
            dropout         = dropout,
            activation      = 'gelu',    # GELU smoother than ReLU for Transformers
            batch_first     = True       # input: (batch, seq, d_model)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        """
        x: (batch, seq_len, input_size)
        """
        x = self.input_proj(x)    # (batch, seq, d_model)
        x = self.pos_enc(x)       # add positional encoding
        x = self.encoder(x)       # self-attention + FFN for each token

        # Mean-pool across the sequence dimension
        # (alternative: take last token, or use a learned [CLS] token)
        x = x.mean(dim=1)         # (batch, d_model)
        return self.head(x).squeeze(-1)   # (batch,)


# Initialise Transformer
trans_model = GNSSTransformer(
    input_size = n_features,
    d_model    = 64,
    nhead      = 4,
    num_layers = 3,
    dim_ff     = 256,
    dropout    = 0.1
).to(device)

# AdamW: Adam + weight decay — recommended for Transformers
trans_opt = torch.optim.AdamW(trans_model.parameters(), lr=5e-4, weight_decay=1e-4)

# CosineAnnealingLR: smoothly decays LR from max to near-zero over T_max epochs
trans_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    trans_opt, T_max=60, eta_min=1e-6
)

total_params_t = sum(p.numel() for p in trans_model.parameters() if p.requires_grad)
print(f'Transformer model:')
print(trans_model)
print(f'\nTotal trainable parameters: {total_params_t:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Transformer Training Loop (same structure as LSTM)
# ─────────────────────────────────────────────────────────────────────────────
TRANS_EPOCHS    = 60
trans_train_ls  = []
trans_val_ls    = []
best_trans_loss = float('inf')
best_trans_path = 'best_trans.pt'

print(f'Training Transformer for {TRANS_EPOCHS} epochs...\n')

for epoch in range(1, TRANS_EPOCHS + 1):
    # Training
    trans_model.train()
    t_loss = 0.0
    for X_b, y_b in train_loader:
        trans_opt.zero_grad()
        pred   = trans_model(X_b.to(device))
        loss   = criterion(pred, y_b.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(trans_model.parameters(), 1.0)  # gradient clipping
        trans_opt.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # Validation
    trans_model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            v_loss += criterion(trans_model(X_b.to(device)), y_b.to(device)).item()
    v_loss /= len(val_loader)

    trans_train_ls.append(t_loss)
    trans_val_ls.append(v_loss)
    trans_sched.step()  # cosine annealing advances every epoch

    if v_loss < best_trans_loss:
        best_trans_loss = v_loss
        torch.save(trans_model.state_dict(), best_trans_path)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{TRANS_EPOCHS} | '
              f'Train MSE: {t_loss:.5f} | '
              f'Val MSE: {v_loss:.5f} | '
              f'LR: {trans_opt.param_groups[0]["lr"]:.2e}')

# Load best checkpoint
trans_model.load_state_dict(torch.load(best_trans_path))
trans_model.eval()

trans_preds = []
with torch.no_grad():
    for X_b, _ in test_loader:
        trans_preds.extend(trans_model(X_b.to(device)).cpu().numpy())
trans_preds = np.array(trans_preds, dtype=np.float32)

trans_rmse = np.sqrt(mean_squared_error(y_te_seq, trans_preds))
trans_mae  = mean_absolute_error(y_te_seq, trans_preds)
trans_r2   = r2_score(y_te_seq, trans_preds)

print(f'\n── Transformer Test Results ──────────────')
print(f'  RMSE : {trans_rmse:.4f} ns')
print(f'  MAE  : {trans_mae:.4f} ns')
print(f'  R²   : {trans_r2:.4f}')

---
## 📐 Section 12: Model 4 — Kalman Filter (Classical Physics-Based Baseline)

> ### What is a Kalman Filter?
> 
> The Kalman Filter (1960) is **the** classical algorithm for estimating noisy dynamic systems. It's not ML — it's an **optimal linear estimator** based on Bayesian probability.
> 
> **Intuition:** Imagine you're tracking a car with GPS (noisy) and an accelerometer (also noisy). The Kalman Filter combines both to get a better estimate than either alone — it knows *how much to trust each sensor* based on their noise characteristics.
> 
> **Why include it here?**
> - It's what real GNSS receivers actually use internally
> - It's the industry benchmark — if your DL model can't beat it, something is wrong
> - It's very fast (microseconds) vs DL (milliseconds)
> 
> **Our clock model state:** `[bias, drift]`
> 
> The state transition says: "next bias = current bias + current drift × Δt"
> 
> **Two steps per timestep:**
> 1. **Predict:** Use physics to predict next state
> 2. **Update:** Correct prediction using the actual measurement
> 
> The **Kalman Gain K** controls the trade-off: high K = trust measurement more; low K = trust prediction more.

In [ ]:
class GNSSClockKalmanFilter:
    """
    Linear Kalman Filter for GNSS satellite clock bias estimation.

    State vector: x = [bias (ns), drift (ns/epoch)]

    Clock model:
        bias(t+1)  = bias(t)  + drift(t)   (constant velocity model)
        drift(t+1) = drift(t)              (drift stays constant)

    This is a 2-state linear system:
        x(t+1) = F × x(t) + process_noise
        z(t)   = H × x(t) + measurement_noise

    Where:
        F = [[1, 1],   (state transition: bias += drift)
             [0, 1]]
        H = [[1, 0]]   (we observe bias only, not drift directly)
    """
    def __init__(self, q_bias=0.05, q_drift=0.001, r_obs=0.5):
        """
        Args:
            q_bias  : process noise variance for bias (how much bias can change randomly)
            q_drift : process noise variance for drift
            r_obs   : observation noise variance (measurement uncertainty)
        """
        # ── State vector (bias, drift) initialised to zero ──────────────────
        self.x = np.array([0.0, 0.0])      # initial state estimate

        # ── State covariance matrix (uncertainty in state estimate) ──────────
        self.P = np.eye(2) * 100.0         # start with high uncertainty

        # ── System matrices ─────────────────────────────────────────────────
        self.F = np.array([[1.0, 1.0],     # state transition
                           [0.0, 1.0]])
        self.H = np.array([[1.0, 0.0]])    # observation (we see only bias)

        # ── Noise matrices ───────────────────────────────────────────────────
        self.Q = np.diag([q_bias, q_drift])  # process noise
        self.R = np.array([[r_obs]])          # observation noise

    def predict(self):
        """Predict next state using the physics model."""
        self.x = self.F @ self.x                          # predicted state
        self.P = self.F @ self.P @ self.F.T + self.Q     # predicted covariance
        return self.x[0]   # return predicted bias

    def update(self, z):
        """
        Update state estimate using an actual measurement.
        Args:
            z: observed clock bias (nanoseconds)
        """
        # Innovation (difference between measurement and prediction)
        innovation = z - self.H @ self.x

        # Innovation covariance
        S = self.H @ self.P @ self.H.T + self.R

        # Kalman Gain: how much to trust the measurement vs prediction
        # K close to 1 → trust measurement; K close to 0 → trust prediction
        K = self.P @ self.H.T @ np.linalg.inv(S)

        # Update state and covariance
        self.x = self.x + (K @ innovation).flatten()
        self.P = (np.eye(2) - K @ self.H) @ self.P


def run_kalman_filter(clock_series, q_bias=0.05, q_drift=0.001, r_obs=0.5):
    """
    Run the Kalman filter over an entire clock bias time series.
    For each step: predict → record prediction → update with actual measurement.
    Returns array of one-step-ahead predictions.
    """
    kf   = GNSSClockKalmanFilter(q_bias=q_bias, q_drift=q_drift, r_obs=r_obs)
    preds = []

    for obs in clock_series:
        pred = kf.predict()    # one-step ahead prediction
        preds.append(pred)     # record BEFORE updating
        kf.update(obs)         # update state with actual measurement

    return np.array(preds)


# Run on full satellite series, evaluate on test portion
clock_series_full = sat_df['clock_ns'].values
kf_preds_full     = run_kalman_filter(clock_series_full)

# Test set = last 15% of data
test_start_idx = int(len(clock_series_full) * 0.85)
kf_test_preds  = kf_preds_full[test_start_idx:]
kf_test_true   = clock_series_full[test_start_idx:]

# Align length (the KF may produce off-by-one vs our feat_df test)
min_len = min(len(kf_test_preds), len(kf_test_true))
kf_test_preds = kf_test_preds[:min_len]
kf_test_true  = kf_test_true[:min_len]

kf_rmse = np.sqrt(mean_squared_error(kf_test_true, kf_test_preds))
kf_mae  = mean_absolute_error(kf_test_true, kf_test_preds)
kf_r2   = r2_score(kf_test_true, kf_test_preds)

print(f'── Kalman Filter Test Results ────────────')
print(f'  RMSE : {kf_rmse:.4f} ns')
print(f'  MAE  : {kf_mae:.4f} ns')
print(f'  R²   : {kf_r2:.4f}')
print(f'\nNote: Kalman is extremely fast (~microseconds per prediction)')
print('It will likely have higher RMSE than DL because it cannot model')
print('the nonlinear components (higher harmonics, space weather effects).')

---
## 📊 Section 13: Model Comparison & Visualisation

> ### Understanding the metrics:
> 
> | Metric | Formula | Interpretation |
> |--------|---------|----------------|
> | **RMSE** | √(mean((y-ŷ)²)) | Average error in nanoseconds — lower is better. Penalises large errors more than MAE |
> | **MAE** | mean(|y-ŷ|) | Average absolute error in nanoseconds — more robust to outliers |
> | **R²** | 1 - SS_res/SS_tot | Fraction of variance explained. R²=1: perfect; R²=0: same as predicting the mean; R²<0: worse than mean |
> 
> **What RMSE means physically:**
> - 1 ns clock error ≈ **30 cm position error**
> - If RMSE = 2 ns → predicted corrections reduce position error by ~60 cm on average
> - IGS real-time products achieve ~0.1 ns → 3 cm accuracy

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Collect all model results into a dictionary for easy comparison
# Note: LSTM and Transformer use SEQ_LEN-trimmed targets (y_te_seq)
#       XGBoost and Kalman use the full test targets (y_test, kf_test_true)
# ─────────────────────────────────────────────────────────────────────────────
n_common = min(len(y_te_seq), len(xgb_pred_test))

results = {
    'Kalman Filter' : {'true': kf_test_true[:n_common],   'pred': kf_test_preds[:n_common]},
    'XGBoost'       : {'true': y_test[:n_common],         'pred': xgb_pred_test[:n_common]},
    'LSTM'          : {'true': y_te_seq[:n_common],       'pred': lstm_preds[:n_common]},
    'Transformer'   : {'true': y_te_seq[:n_common],       'pred': trans_preds[:n_common]},
}

# Compute metrics for all models
metrics_list = []
for name, r in results.items():
    rmse = np.sqrt(mean_squared_error(r['true'], r['pred']))
    mae  = mean_absolute_error(r['true'], r['pred'])
    r2   = r2_score(r['true'], r['pred'])
    metrics_list.append({
        'Model': name,
        'RMSE (ns)': round(rmse, 4),
        'MAE (ns)': round(mae, 4),
        'R²': round(r2, 4),
        'Position error (cm)': round(rmse * 30, 1)  # 1 ns = 30 cm
    })

metrics_df = pd.DataFrame(metrics_list).sort_values('RMSE (ns)')
print('='*65)
print('MODEL COMPARISON — Test Set Results')
print('='*65)
print(metrics_df.to_string(index=False))
print('='*65)
print(f'Reminder: 1 ns clock error ≈ 30 cm positioning error')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1: Bar chart comparison
# ─────────────────────────────────────────────────────────────────────────────
colors = ['#FF9800', '#2196F3', '#4CAF50', '#9C27B0']  # orange, blue, green, purple
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Model Comparison — Test Set Performance', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, ['RMSE (ns)', 'MAE (ns)', 'R²']):
    vals = metrics_df[metric].values
    bars = ax.bar(metrics_df['Model'], vals, color=colors, width=0.55, edgecolor='white')
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
    # Add value labels on top of bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2: Time-series predictions overlay (first 200 test epochs)
# ─────────────────────────────────────────────────────────────────────────────
PLOT_N  = min(200, n_common)
x_idx   = np.arange(PLOT_N)
model_colors = ['#FF9800', '#2196F3', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
fig.suptitle(f'Satellite {TARGET_PRN} — Clock Bias Prediction (first {PLOT_N} test epochs)',
             fontsize=13, fontweight='bold')

for ax, (name, r), col in zip(axes, results.items(), model_colors):
    true = r['true'][:PLOT_N]
    pred = r['pred'][:PLOT_N]
    rmse = np.sqrt(mean_squared_error(r['true'], r['pred']))

    # True signal
    ax.plot(x_idx, true, label='True', lw=1.4, color='#222', zorder=3)
    # Predicted signal
    ax.plot(x_idx, pred, label=f'{name} (RMSE={rmse:.4f} ns)',
            lw=1.1, color=col, linestyle='--', alpha=0.9, zorder=2)
    # Shade the error region
    ax.fill_between(x_idx, true, pred, alpha=0.15, color=col)

    ax.set_ylabel('Clock bias (ns)', fontsize=10)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_title(name, fontsize=11)

axes[-1].set_xlabel('Test epoch (each = 15 minutes)', fontsize=10)
plt.tight_layout()
plt.savefig('predictions_overlay.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3: Residual analysis — are errors Gaussian? Biased? Time-dependent?
# Good residuals should be: zero-mean, Gaussian, no obvious patterns over time
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Residual Analysis (true − predicted)', fontsize=13, fontweight='bold')

for col_idx, (name, r) in enumerate(results.items()):
    res = r['true'] - r['pred']

    # Row 0: Residuals over time
    axes[0, col_idx].plot(res[:300], lw=0.7, color=model_colors[col_idx])
    axes[0, col_idx].axhline(0, color='black', lw=0.8, linestyle='--')
    axes[0, col_idx].set_title(name, fontsize=10)
    axes[0, col_idx].set_ylabel('Residual (ns)' if col_idx == 0 else '')
    axes[0, col_idx].set_xlabel('Test epoch')

    # Row 1: Residual histogram
    axes[1, col_idx].hist(res, bins=50, color=model_colors[col_idx],
                          alpha=0.8, edgecolor='white')
    axes[1, col_idx].axvline(0, color='red', lw=1.2, linestyle='--')
    axes[1, col_idx].set_title(f'μ={np.mean(res):.3f}, σ={np.std(res):.3f} ns', fontsize=9)
    axes[1, col_idx].set_xlabel('Residual (ns)')
    axes[1, col_idx].set_ylabel('Count' if col_idx == 0 else '')

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('What to look for:')
print('  ✅ Good: residuals centred at 0, Gaussian histogram, no obvious time patterns')
print('  ❌ Bad : systematic drift in residuals → model not capturing the trend')
print('  ❌ Bad : heavy tails in histogram → model misses large errors')

---
## 🔍 Section 14: XGBoost Feature Importance

> **Feature importance** tells us which features XGBoost found most useful. This gives us insight into the underlying problem structure.
> 
> Expected findings:
> - **`lag_1`** should dominate (GNSS clocks are very predictable from the immediate past)
> - **`ema_8`** or **`roll_mean_4`** should rank high (short-term average captures trend)
> - **Fourier features** (`sin_1`, `cos_1`) should appear — confirming the orbital periodicity
> - **`kp_index`** may appear — solar weather does affect clock errors
> 
> If `lag_1` completely dominates (>80% importance), the signal might be too smooth and we could achieve good predictions with a simple persistence model (just predict `clock(t+1) = clock(t)`). In that case, RMSE should be compared against this naive baseline!

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# XGBoost built-in feature importance (gain-based)
# 'gain' = average improvement in loss when a feature is used for splitting
# ─────────────────────────────────────────────────────────────────────────────
fi_series = pd.Series(
    xgb_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

top_n = 25
fig, ax = plt.subplots(figsize=(10, 6))
fi_top = fi_series.head(top_n)

# Color code by feature type
def get_color(feat_name):
    if 'lag' in feat_name:    return '#2196F3'   # blue = lag features
    if 'sin' in feat_name or 'cos' in feat_name: return '#9C27B0'  # purple = Fourier
    if 'roll' in feat_name or 'ema' in feat_name: return '#4CAF50' # green = rolling stats
    if 'diff' in feat_name:   return '#FF9800'   # orange = differences
    if 'kp' in feat_name:     return '#F44336'   # red = space weather
    return '#607D8B'                              # gray = time features

bar_colors = [get_color(f) for f in fi_top.index]
ax.barh(fi_top.index[::-1], fi_top.values[::-1], color=bar_colors[::-1])
ax.set_xlabel('Feature importance (gain)')
ax.set_title(f'XGBoost — Top {top_n} Feature Importances', fontsize=12)

# Legend
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#2196F3', label='Lag features'),
    Patch(facecolor='#9C27B0', label='Fourier/cyclical'),
    Patch(facecolor='#4CAF50', label='Rolling statistics'),
    Patch(facecolor='#FF9800', label='Differences'),
    Patch(facecolor='#F44336', label='Space weather (Kp)'),
]
ax.legend(handles=legend_els, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Top 10 features:')
for i, (feat, imp) in enumerate(fi_series.head(10).items(), 1):
    print(f'  {i:2d}. {feat:<25} importance = {imp:.6f}')

---
## 📈 Section 15: Training Loss Curves

> **Reading loss curves is a core skill!** Here's what to look for:
> 
> - **Healthy training:** Both train and val loss decrease steadily, then plateau
> - **Overfitting:** Train loss keeps decreasing but val loss increases → model memorising training data
> - **Underfitting:** Both losses are high and don't converge → model is too simple or LR is wrong
> - **Learning rate too high:** Loss oscillates wildly or explodes
> - **Learning rate too low:** Loss decreases very slowly (takes forever to converge)
> 
> The **gap between train and val loss** tells you the generalisation gap. For GNSS data, a small gap is expected because the signal is deterministic (physics-based), unlike say stock prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training Loss Curves', fontsize=13, fontweight='bold')

# LSTM
epochs_lstm = range(1, len(lstm_train_losses) + 1)
axes[0].plot(epochs_lstm, lstm_train_losses, label='Train loss', color='steelblue', lw=1.4)
axes[0].plot(epochs_lstm, lstm_val_losses,   label='Val loss',   color='coral', lw=1.4)
axes[0].set_title('LSTM', fontsize=11)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE loss')
axes[0].legend()
# Annotate minimum val loss
best_ep = np.argmin(lstm_val_losses)
axes[0].axvline(best_ep + 1, color='gray', lw=1, linestyle=':', label='Best checkpoint')
axes[0].text(best_ep + 2, max(lstm_val_losses)*0.9,
             f'Best: epoch {best_ep+1}', fontsize=8, color='gray')

# Transformer
epochs_trans = range(1, len(trans_train_ls) + 1)
axes[1].plot(epochs_trans, trans_train_ls, label='Train loss', color='purple', lw=1.4)
axes[1].plot(epochs_trans, trans_val_ls,   label='Val loss',   color='coral', lw=1.4)
axes[1].set_title('Transformer', fontsize=11)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🔮 Section 16: Multi-Step Forecasting (24-Hour Lookahead)

> ### What is multi-step (autoregressive) forecasting?
> 
> So far we've been doing **1-step-ahead prediction** — given past data, predict the very next value. But in practice, GNSS corrections need to work for extended periods (think: a drone mission, a ship voyage, or periods when ground contact is lost).
> 
> **Strategy: Iterative (autoregressive) forecasting**
> ```
> At time T, we have window [T-31, ..., T]
> Predict T+1 → append to window → now predict T+2
> Repeat for 96 steps = 24 hours
> ```
> 
> **Important caveat:** Error accumulates! Each prediction uses the previous (imperfect) prediction as input. This is called **error propagation** and is why long-horizon forecasting is harder.
> 
> For GNSS specifically, this is called **satellite clock prediction** — GNSS receivers need predicted corrections when they can't communicate with reference stations.

In [ ]:
def iterative_forecast_lstm(model, last_window, n_steps=96):
    """
    Generate n_steps ahead forecasts autoregressively using the LSTM.

    Strategy:
        1. Start with the last SEQ_LEN window from the test set
        2. Predict next value
        3. Shift window forward: drop oldest row, append a new row
           (we use the last row's features and update the lag features)
        4. Repeat n_steps times

    Args:
        model       : trained LSTM or Transformer (eval mode)
        last_window : numpy array of shape (SEQ_LEN, n_features) — the seed window
        n_steps     : number of future steps to predict (96 × 15min = 24h)

    Returns:
        numpy array of shape (n_steps,) containing forecasted clock bias
    """
    model.eval()
    window    = last_window.copy()   # (SEQ_LEN, n_features)
    forecasts = []

    with torch.no_grad():
        for step in range(n_steps):
            # Reshape to (1, SEQ_LEN, n_features) for batch dimension
            inp = torch.FloatTensor(window).unsqueeze(0).to(device)

            # Get prediction (in scaled space)
            pred_scaled = model(inp).item()
            forecasts.append(pred_scaled)

            # Update window: shift left (drop oldest), append new step
            # Simplification: copy the last row's features and update
            # the first feature slot (which holds the scaled clock value)
            # In a full system, you'd recompute ALL lag/rolling features
            new_row              = window[-1].copy()
            new_row[0]           = pred_scaled   # update first feature with prediction
            window               = np.vstack([window[1:], new_row])  # slide window

    return np.array(forecasts)


# Use the last test window as the seed
seed_window = X_te_seq[-1]   # shape: (SEQ_LEN, n_features)

print('Generating 24-hour (96 × 15 min) iterative forecasts...')
lstm_forecast  = iterative_forecast_lstm(lstm_model,  seed_window, n_steps=96)
trans_forecast = iterative_forecast_lstm(trans_model, seed_window, n_steps=96)

# ── Plot: recent history + forecast ─────────────────────────────────────────
recent_n    = 200
recent_true = y_te_seq[-recent_n:]
x_hist      = np.arange(recent_n)
x_fore      = np.arange(recent_n, recent_n + 96)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(x_hist, recent_true, label='Observed (test)', color='#333', lw=1.4, zorder=4)
ax.plot(x_fore, lstm_forecast,  label='LSTM 24h forecast',        color='steelblue', lw=1.2, linestyle='--')
ax.plot(x_fore, trans_forecast, label='Transformer 24h forecast', color='purple',    lw=1.2, linestyle='-.')

# Vertical line separating observed from forecast
ax.axvline(recent_n, color='gray', lw=1.2, linestyle=':', label='Forecast start')
ax.fill_betweenx(
    [recent_true.min() * 1.1, recent_true.max() * 1.1],
    recent_n, recent_n + 96,
    alpha=0.05, color='gray', label='Forecast window'
)

ax.set_title(f'Satellite {TARGET_PRN} — 24-hour clock bias forecast', fontsize=13)
ax.set_xlabel('Epoch (each = 15 minutes)')
ax.set_ylabel('Clock bias (nanoseconds)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('24h_forecast.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nNote: Error grows with forecast horizon (error propagation).')
print('After ~4 hours the forecast uncertainty typically exceeds the Kalman filter.')

---
## 🧾 Section 17: Naive Baseline & Final Summary

> ### Always compare against naive baselines!
> 
> Before concluding your ML model is good, compare it against the simplest possible prediction:
> 
> **Persistence baseline:** `clock(t+1) = clock(t)` (just predict the current value)
> 
> If a highly autocorrelated signal (like GNSS clocks) has RMSE of 1.0 ns with the persistence model, your DL model should do significantly better — otherwise, why bother with the complexity?
> 
> A good rule: **your model should beat the baseline by at least 20-30%** to be considered useful in practice.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Naive persistence baseline: predict clock(t+1) = clock(t)
# This is the simplest possible model and a mandatory comparison
# ─────────────────────────────────────────────────────────────────────────────
test_clock_series = sat_df['clock_ns'].values[val_end:]
if len(test_clock_series) > 1:
    naive_true = test_clock_series[1:]    # actual values at t+1
    naive_pred = test_clock_series[:-1]   # prediction = value at t
    naive_rmse = np.sqrt(mean_squared_error(naive_true, naive_pred))
    naive_mae  = mean_absolute_error(naive_true, naive_pred)
    naive_r2   = r2_score(naive_true, naive_pred)
    print(f'── Naive Persistence Baseline ────────────────────────────')
    print(f'   Predict clock(t+1) = clock(t)')
    print(f'   RMSE : {naive_rmse:.4f} ns  ({naive_rmse*30:.1f} cm position error)')
    print(f'   MAE  : {naive_mae:.4f} ns')
    print(f'   R²   : {naive_r2:.4f}')

# ─────────────────────────────────────────────────────────────────────────────
# Final comprehensive summary
# ─────────────────────────────────────────────────────────────────────────────
print()
print('='*65)
print('FINAL RESULTS SUMMARY')
print('='*65)
print(f'{"Model":<22} {"RMSE (ns)":>10} {"MAE (ns)":>10} {"R²":>8}')
print('-'*65)
if len(test_clock_series) > 1:
    print(f'{"Naive Persistence":<22} {naive_rmse:>10.4f} {naive_mae:>10.4f} {naive_r2:>8.4f}')
for _, row in metrics_df.iterrows():
    print(f'{row["Model"]:<22} {row["RMSE (ns)"]:>10.4f} {row["MAE (ns)"]:>10.4f} {row["R²"]:>8.4f}')
print('='*65)
print()
best_model = metrics_df.iloc[0]['Model']
best_rmse  = metrics_df.iloc[0]['RMSE (ns)']
print(f'🏆 Best performing model: {best_model} (RMSE = {best_rmse:.4f} ns)')
print(f'   Equivalent position accuracy: ±{best_rmse * 30:.1f} cm')
if len(test_clock_series) > 1:
    improvement = (naive_rmse - best_rmse) / naive_rmse * 100
    print(f'   Improvement over persistence baseline: {improvement:.1f}%')

---
## 📚 Section 18: Key Takeaways & What to Do Next

### What we built
A complete ML pipeline for GNSS satellite clock error prediction:
- Data ingestion from a real international geodetic archive
- Principled feature engineering grounded in GNSS physics
- Four models: Kalman Filter → XGBoost → LSTM → Transformer
- Rigorous evaluation with multiple metrics and residual analysis
- 24-hour autoregressive forecasting

### Key lessons for DL students

| Lesson | What we saw |
|--------|-------------|
| Feature engineering matters as much as model architecture | Lag features + Fourier terms gave XGBoost competitive performance |
| Always baseline | Persistence model gives a reality check on "how hard is this problem" |
| Chronological splitting is mandatory for time series | Random split = data leakage = fake results |
| Gradient clipping saves RNN training | Without it, LSTM gradients explode on long sequences |
| Bidirectionality helps for periodic signals | The backward pass recognises orbital phase from future context |
| Kalman Filter is hard to beat for 1-step prediction | It's optimal for linear Gaussian systems — DL wins only when nonlinearity matters |

### Suggested extensions for your project report
1. **Real data:** Register at [NASA CDDIS](https://cddis.nasa.gov) — free, approval in 1-2 days
2. **Multi-satellite model:** Add PRN as an embedding → one model for all satellites
3. **RINEX nav parsing:** Compute broadcast ephemeris errors (subtract from SP3)
4. **Hyperparameter optimisation:** Use Optuna for automated tuning
5. **Ensemble:** Weighted average of LSTM + Transformer → usually beats either alone
6. **Attention visualisation:** Plot attention weights of the Transformer → which past timesteps matter most?
7. **Anomaly detection:** Flag unusual clock behaviour (satellite maneuvers, hardware issues)

### SDG impact quantification (for your report)
- Current IGS real-time corrections: ~0.5-2 ns accuracy
- Commercial RTK GNSS requires ~1 ns corrections for centimeter-level accuracy
- Better predictions → fewer ground station dependencies → more accessible in developing regions (SDG 9 + 17)
- Applications: precision agriculture, autonomous vehicles, emergency response, climate sensors

### References
- Kouba & Héroux (2001). "Precise Point Positioning Using IGS Orbit and Clock Products" — the foundational paper on IGS corrections
- IS-GPS-200 (2022). GPS Interface Control Document — satellite signal specification
- Hochreiter & Schmidhuber (1997). "Long Short-Term Memory" — the original LSTM paper
- Vaswani et al. (2017). "Attention Is All You Need" — the Transformer paper
- IGS website: [igs.org](https://www.igs.org) — real-time data access
- NASA CDDIS: [cddis.nasa.gov](https://cddis.nasa.gov) — SP3 archive